In [11]:
import pandas as pd
import numpy as np

In [12]:
df = pd.read_csv(
    "../prepared_data/reduced_vars_with_hmm.csv",
    index_col=0,
    parse_dates=True
)

In [13]:
TARGET_COL = "y_SP500_bin_4w"

y = df[TARGET_COL].copy()

X = df.drop(columns=[TARGET_COL], errors="ignore").copy()
X = X.drop(columns=["SP500"], errors="ignore")

X = X.select_dtypes(include=[np.number]).copy()
X = X.replace([np.inf, -np.inf], np.nan)

data = X.join(y.rename("target")).dropna()
X = data.drop(columns=["target"])
y = data["target"]

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts(dropna=False))

X shape: (1078, 36)
y value counts:
 target
1.0    857
0.0    221
Name: count, dtype: int64


In [14]:
if y.dtype == "object":
    y_mapped = y.map({"IN": 0, "OUT": 1})
    if y_mapped.isna().any():
        raise ValueError(f"Valores inesperados en y: {y.unique()}")
    y = y_mapped.astype(int)
else:
    if y.dtype == "bool":
        y = y.astype(int)
    elif np.issubdtype(y.dtype, np.number):
        if np.all(np.isclose(y.values, y.values.astype(int))):
            y = y.astype(int)

print("y dtype:", y.dtype, "| unique:", np.unique(y))
print("Counts [IN=0, OUT=1]:", np.bincount(y))

y dtype: int32 | unique: [0 1]
Counts [IN=0, OUT=1]: [221 857]


In [15]:
CUTOFF_DATE = "2024-01-01"

X_train = X.loc[X.index < CUTOFF_DATE].copy()
X_test  = X.loc[X.index >= CUTOFF_DATE].copy()

y_train = y.loc[y.index < CUTOFF_DATE].copy()
y_test  = y.loc[y.index >= CUTOFF_DATE].copy()

print("Train:", X_train.index.min(), "->", X_train.index.max(), "| n =", len(X_train))
print("Test :", X_test.index.min(), "->", X_test.index.max(), "| n =", len(X_test))
print("y_train counts:\n", y_train.value_counts())
print("y_test counts:\n", y_test.value_counts())

Train: 2005-04-08 00:00:00 -> 2023-12-29 00:00:00 | n = 978
Test : 2024-01-05 00:00:00 -> 2025-11-28 00:00:00 | n = 100
y_train counts:
 target
1    771
0    207
Name: count, dtype: int64
y_test counts:
 target
1    86
0    14
Name: count, dtype: int64


### MODEL: NEURAL NETWORK (MLP)

In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    f1_score
)

np.random.seed(42)

In [17]:
counts = y_train.value_counts().sort_index()

if 0 not in counts.index or 1 not in counts.index:
    raise ValueError("y_train debe tener clases 0 y 1. Revisa el encoding.")

w0 = len(y_train) / (2 * counts.loc[0])
w1 = len(y_train) / (2 * counts.loc[1])

w0, w1 = float(w0), float(w1)
class_weight = {0: w0, 1: w1}

print("Class weights -> IN(0):", round(w0, 4), "OUT(1):", round(w1, 4))

sw_all = y_train.map({0: w0, 1: w1}).values

Class weights -> IN(0): 2.3623 OUT(1): 0.6342


In [18]:
tscv = TimeSeriesSplit(n_splits=3)

In [19]:
param_grid = {
    "hidden_layer_sizes": [(16,), (32,), (16, 8)],
    "alpha": [0.01, 0.1, 1.0],
    "learning_rate_init": [1e-3, 5e-4]
}

n_combos = len(list(ParameterGrid(param_grid)))
print("Grid combinations:", n_combos)

Grid combinations: 18


In [20]:
# Scaler global para el grid search
scaler_gs = StandardScaler()
X_train_s = scaler_gs.fit_transform(X_train)

best_score = -1
best_params = None

for params in ParameterGrid(param_grid):
    fold_scores = []

    for tr_idx, val_idx in tscv.split(X_train_s):
        X_tr, X_val = X_train_s[tr_idx], X_train_s[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        sw_tr = sw_all[tr_idx]

        model = MLPClassifier(
            hidden_layer_sizes=params["hidden_layer_sizes"],
            alpha=params["alpha"],
            learning_rate_init=params["learning_rate_init"],
            activation="relu",
            solver="adam",
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=42
        )

        model.fit(X_tr, y_tr)

        proba = model.predict_proba(X_val)[:, 1]
        pred = (proba >= 0.5).astype(int)
        fold_scores.append(balanced_accuracy_score(y_val, pred))

    mean_score = float(np.mean(fold_scores))

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print("Best CV balanced_accuracy:", round(best_score, 4))
print("Best params:", best_params)

Best CV balanced_accuracy: 0.5907
Best params: {'alpha': 1.0, 'hidden_layer_sizes': (16,), 'learning_rate_init': 0.001}


In [21]:
best_nn = MLPClassifier(
    hidden_layer_sizes=best_params["hidden_layer_sizes"],
    alpha=best_params["alpha"],
    learning_rate_init=best_params["learning_rate_init"],
    activation="relu",
    solver="adam",
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

best_nn.fit(X_train_s, y_train)

MLPClassifier(alpha=1.0, early_stopping=True, hidden_layer_sizes=(16,),
              max_iter=500, random_state=42)

In [22]:
val_ratio = 0.2
split_val = int(len(X_train) * (1 - val_ratio))

X_tr, X_val = X_train.iloc[:split_val], X_train.iloc[split_val:]
y_tr, y_val = y_train.iloc[:split_val], y_train.iloc[split_val:]

scaler_tmp = StandardScaler()
X_tr_s = scaler_tmp.fit_transform(X_tr)
X_val_s = scaler_tmp.transform(X_val)

sw_tr = y_tr.map({0: w0, 1: w1}).values

tmp = MLPClassifier(
    hidden_layer_sizes=best_params["hidden_layer_sizes"],
    alpha=best_params["alpha"],
    learning_rate_init=best_params["learning_rate_init"],
    activation="relu",
    solver="adam",
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

tmp.fit(X_tr_s, y_tr)

val_proba = tmp.predict_proba(X_val_s)[:, 1]

thresholds = np.linspace(0.35, 0.60, 51)

best_thr = 0.5
best_f1_down = -1

for thr in thresholds:
    val_pred = (val_proba >= thr).astype(int)
    f1_down = f1_score(y_val, val_pred, pos_label=0)
    if f1_down > best_f1_down:
        best_f1_down = f1_down
        best_thr = float(thr)

print("Best threshold:", round(best_thr, 3), "| Best F1 DOWN:", round(best_f1_down, 4))

Best threshold: 0.395 | Best F1 DOWN: 0.4242


In [23]:
# =====================================================
# Test analytics (2024 onward) — BINARY
# Model: Neural Network (best_nn)
# 0 = OUT, 1 = IN
# =====================================================

X_test_s = scaler_gs.transform(X_test)

test_proba = best_nn.predict_proba(X_test_s)[:, 1]
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== Neural Network MLP (TEST) ===")
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Balanced Acc:", round(balanced_accuracy_score(y_test, test_pred), 4))
print("F1 (IN):", round(f1_score(y_test, test_pred, pos_label=1), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["True OUT (0)", "True IN (1)"],
    columns=["Pred OUT (0)", "Pred IN (1)"]
)

print("\nConfusion Matrix:")
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test, test_pred, target_names=["OUT (0)", "IN (1)"]))


=== Neural Network MLP (TEST) ===
Accuracy: 0.8
Balanced Acc: 0.5548
F1 (IN): 0.8851
ROC-AUC: 0.6628

Confusion Matrix:
              Pred OUT (0)  Pred IN (1)
True OUT (0)             3           11
True IN (1)              9           77

Classification Report:
              precision    recall  f1-score   support

     OUT (0)       0.25      0.21      0.23        14
      IN (1)       0.88      0.90      0.89        86

    accuracy                           0.80       100
   macro avg       0.56      0.55      0.56       100
weighted avg       0.79      0.80      0.79       100



#### Metrics

In [24]:
p_tr = best_nn.predict_proba(X_train_s)[:, 1]
pred_tr = (p_tr >= best_thr).astype(int)

pred_va = (val_proba >= best_thr).astype(int)

p_te = test_proba
pred_te = test_pred

def r4(x):
    return float(f"{x:.4f}")

print(f"\n=== QUICK METRICS (0=IN, 1=OUT) @ thr={best_thr:.3f} ===")
print("TRAIN  acc:", r4(accuracy_score(y_train, pred_tr)),
      "bal_acc:", r4(balanced_accuracy_score(y_train, pred_tr)),
      "F1_OUT:", r4(f1_score(y_train, pred_tr, pos_label=1)))

print("VAL    acc:", r4(accuracy_score(y_val, pred_va)),
      "bal_acc:", r4(balanced_accuracy_score(y_val, pred_va)),
      "F1_OUT:", r4(f1_score(y_val, pred_va, pos_label=1)))

print("TEST   acc:", r4(accuracy_score(y_test, pred_te)),
      "bal_acc:", r4(balanced_accuracy_score(y_test, pred_te)),
      "F1_OUT:", r4(f1_score(y_test, pred_te, pos_label=1)))


=== QUICK METRICS (0=IN, 1=OUT) @ thr=0.395 ===
TRAIN  acc: 0.8037 bal_acc: 0.5433 F1_OUT: 0.8888
VAL    acc: 0.6122 bal_acc: 0.652 F1_OUT: 0.7077
TEST   acc: 0.8 bal_acc: 0.5548 F1_OUT: 0.8851


### Data extraction

In [25]:
# =====================================================
# DF for trading sim (2024+ only) + simple checks + save
# Model: Neural Network (best_nn)
# =====================================================

df_trading = df.loc[df.index >= CUTOFF_DATE].copy()

if not df_trading.index.equals(X_test.index):
    print("WARNING: df_trading.index != X_test.index")
    print("df_trading:", df_trading.index.min(), "->", df_trading.index.max(), "n=", len(df_trading))
    print("X_test    :", X_test.index.min(),     "->", X_test.index.max(),     "n=", len(X_test))
    missing_in_df = X_test.index.difference(df_trading.index)
    missing_in_X  = df_trading.index.difference(X_test.index)
    print("Missing in df_trading (should be 0):", len(missing_in_df))
    print("Missing in X_test (should be 0):", len(missing_in_X))

df_trading["p_out"] = pd.Series(test_proba, index=X_test.index)
df_trading["pred_out"] = pd.Series(test_pred, index=X_test.index)
df_trading["y_out_true"] = pd.Series(y_test, index=X_test.index)

nan_counts = df_trading[["p_out", "pred_out", "y_out_true"]].isna().sum()
print("\nNaNs in key cols:\n", nan_counts)

print("\nRows in df_trading:", len(df_trading))
print("Pred rows (X_test):", len(X_test))
print("All key cols non-null? ->", (nan_counts.sum() == 0))

out_path = "../predictions/nn_preds.csv"
df_trading.to_csv(out_path)
print("\nSaved ->", out_path)


NaNs in key cols:
 p_out         0
pred_out      0
y_out_true    0
dtype: int64

Rows in df_trading: 100
Pred rows (X_test): 100
All key cols non-null? -> True

Saved -> ../predictions/nn_preds.csv
